<a href="https://colab.research.google.com/github/RMoulla/IAO_Syn/blob/main/TP_RAG_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TP : Retrieval Augmented Generation (RAG)**

Dans ce TP, nous allons construire un **Retrieval Augmented Generation (RAG)**. L'objectif est de fournir une méthode complète pour traiter un fichier PDF, en extraire des informations pertinentes, puis les utiliser pour répondre à une requête utilisateur de manière augmentée. Ce TP est divisé en trois phases distinctes :

1. **Extraction et segmentation du texte** : Vous allez apprendre à extraire du texte à partir d'un fichier PDF et à le segmenter en sections de taille définie.
2. **Calcul des embeddings et recherche des segments similaires** : Nous utiliserons des techniques d'embeddings pour trouver les sections du texte les plus pertinentes par rapport à une requête donnée.
3. **Génération de réponses avec un modèle de type ChatGPT** : En utilisant le contexte extrait et le modèle GPT-4, nous allons générer une réponse contextualisée à la question posée par l'utilisateur.

Par ailleurs, nous allons utiliser des outils comme `pdfplumber`, pour parser les documents pdf, `sentence-transformers` pour les embeddings et l'API OpenAI pour générer des réponses augmentées par le contexte.

In [5]:
!pip install pdfplumber

## **Étape 1 : Extraction et segmentation du texte**

Dans cette première étape, nous allons nous concentrer sur l'extraction du texte d'un fichier PDF et sa segmentation en blocs de taille définie. L'objectif est de rendre le texte extrait plus facile à traiter dans les phases suivantes.

### Détails :
1. **Extraction du texte** : À l'aide de `pdfplumber`, vous allez extraire tout le texte d'un fichier PDF donné. Cette étape vous permettra d'obtenir une version brute du contenu du document, qui servira de base pour les autres étapes.
   
2. **Segmentation du texte** : Une fois le texte extrait, vous le diviserez en segments de taille fixe (par exemple, 500 caractères par segment) à l'aide de la fonction `textwrap.wrap()` de Python. Cela permet de découper le texte en morceaux plus faciles à manipuler dans la phase suivante où nous calculerons les similarités.

Cette étape est cruciale, car elle permet de structurer le texte pour le rendre exploitable par la suite. La qualité de l'extraction et de la segmentation influencera directement les résultats des phases suivantes.

In [6]:
import pdfplumber
import textwrap

# Chemin vers le fichier PDF
pdf_path = "Cours.pdf"

# Extraction du texte et des tables
full_text = ""
tables = []  # Liste pour stocker les tables extraites

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        # Extraction du texte
        page_text = page.extract_text()
        if page_text:
            full_text += page_text

        # Extraction des tables
        page_tables = page.extract_tables()
        if page_tables:
            tables.extend(page_tables)  # Ajouter les tables extraites à la liste

# Segmenter le texte en utilisant textwrap.wrap avec une longueur maximale de caractères

paragraphs = textwrap.wrap(full_text.strip(), width=500)

# Affichage des segments extraits
print(f"Nombre de segments de texte extraits : {len(paragraphs)}")
for i, para in enumerate(paragraphs[:5], 1):
    print(f"Segment {i}: {para}\n")

# Affichage des tables extraites
if tables:
    print(f"Nombre de tables extraites : {len(tables)}")
    print("Première table extraite :")
    for row in tables[0]:
        print(row)
else:
    print("Aucune table trouvée dans le PDF.")

Nombre de segments de texte extraits : 90
Segment 1: L’IAG pour chefs de projets, analystes, développeurs et architectes SI Redha Moulla Paris, 5-6 novembre 2026Plan de la formation • Qu’est-ce que l’intelligence artificielle? • Eléments de machine learning et de deep learning • Large Language Models (LLM) • Les bonnes pratiques de prompting • Les RAG et les Agents • L’IAG dans le cycle de développement logiciel • Enjeux éthiques, de sécurité et de conformité de l’IAG 2 / 104Qu’est-ce que l’intelligence artificielle ?Définition littérale de

Segment 2: l’intelligence artificielle 1. Intelligence Ensemble des fonctions mentales ayant pour objet la connaissance conceptuelle et relationnelle. - Larousse 2. Artificielle Qui est produit de l’activité humaine (opposé à la nature). - Larousse 4 / 104Qu’est-ce que l’intelligence? La notion d’intelligence recouvre plusieurs facultés cognitives : 1 Raisonnement : La capacité à résoudre des problèmes et à faire des déductions logiques. 2 Apprenti

## **Étape 2 : Calcul des Embeddings et recherche des segments similaires**

Dans cette deuxième étape, nous allons calculer des **embeddings** pour chaque segment de texte extrait et effectuer une recherche pour identifier les segments les plus similaires à une requête utilisateur donnée.

### Détails :
1. **Calcul des embeddings** : Nous allons utiliser la bibliothèque `sentence-transformers` pour générer des vecteurs d’embeddings pour chaque segment de texte. Ces vecteurs capturent les caractéristiques sémantiques des segments, permettant ainsi de comparer leur pertinence par rapport à une requête donnée.

2. **Recherche des segments les plus similaires** : Une fois les embeddings calculés, nous allons utiliser la **similarité cosinus** pour mesurer la proximité entre l’embedding de la requête utilisateur et les embeddings des segments de texte. Les segments les plus proches seront sélectionnés pour la prochaine étape.

3. **Sélection des top-N segments** : Nous sélectionnerons les `n` segments les plus similaires à la requête, qui serviront de contexte pour générer une réponse augmentée dans la phase suivante.

Cette étape est essentielle pour filtrer le contenu extrait en fonction de sa pertinence par rapport à la requête utilisateur. Elle permet de s'assurer que seuls les segments les plus utiles sont utilisés pour générer une réponse cohérente et contextuelle.

In [7]:
!pip install sentence_transformers

In [8]:
from sentence_transformers import SentenceTransformer, util

# Charger un modèle pré-entraîné de sentence-transformers
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
# Calcul des embeddings pour chaque segmnt de texte (tableaux NumPy par défaut)
embeddings = model.encode(paragraphs)

In [10]:
len(embeddings[0])

384

In [11]:

# Requête utilisateur pour laquelle nous cherchons les segments similaires
query = "Quels sont les LLMs open source ?"
query_embedding = model.encode(query)

# Calcul des similarités cosinus entre la requête et les segments
similarities = util.cos_sim(query_embedding, embeddings)

# Sélection des top-N documents les plus similaires
top_n = 10
top_results = similarities.topk(top_n)[1]  # Indices des top-N documents similaires

# Accéder aux indices des résultats top-N
top_n_paragraphs = [paragraphs[i] for i in top_results[0]]

# Affichage des top-N segments les plus similaires
for i, para in enumerate(top_n_paragraphs, 1):
    print(f"Segment {i} similaire : {para}\n")

Segment 1 similaire : personnelles. • RGPD : droit à l’oubli, consentement, minimisation des données. • Localisation des données : datacenters UE vs hors UE. • Solutions : • anonymisation avant envoi, • hébergement de modèles open source en interne, • passage par un proxy d’entreprise. 97 / 104Propriété intellectuelle et IA générative • Données d’entraînement : l’utilisation d’œuvres protégées soulève des questions de respect des droits, de transparence sur les sources et de rémunération des créateurs. • Protection

Segment 2 similaire : Réponse Question Identifier les Formuler des Vérifier la Formuler la à résoudre sous-problèmes hypothèses cohérence réponse Réviser une piste Davantage de calcul peut améliorer la résolution des problèmes complexes, au prix d’un temps de réponse plus long. 51 / 104LLMs propriétaires vs Open Source • Modèles propriétaires • ChatGPT, Claude, Gemini, Grok... • Poids non disponibles au public. • Performances élevées, accès limité par API payante • Protecti

In [12]:
len(similarities)

1

## **Étape 3 : Génération de réponses avec ChatGPT**

Dans cette troisième, nous allons utiliser les segments de texte sélectionnés et une requête utilisateur pour générer une réponse contextuelle en utilisant l'API OpenAI, avec un modèle de type **ChatGPT**.

### Objectifs :
- Utiliser le modèle **GPT-4** pour générer une réponse basée sur les segments de texte les plus similaires à la requête.
- Envoyer les segments sélectionnés et la requête utilisateur sous forme de **messages** à l’API ChatGPT.
- Obtenir une réponse contextuelle, augmentée par les informations pertinentes extraites du texte.

### Détails :
1. **Création du contexte** : Nous allons concaténer les segments sélectionnés lors de l’étape précédente afin de créer un contexte cohérent à transmettre au modèle GPT-4. Ce contexte servira à fournir un maximum d’informations pertinentes pour générer une réponse précise.

2. **Appel à l’API ChatGPT** : En utilisant le contexte et la requête de l’utilisateur, nous formulerons un prompt que nous enverrons à l’API ChatGPT. Le prompt sera structuré sous forme de messages (avec les rôles "system" et "user") pour que le modèle comprenne le contexte de la conversation.

3. **Génération et récupération de la réponse** : Le modèle GPT-4 générera une réponse basée sur le contexte fourni. La réponse sera ensuite récupérée et affichée comme résultat final.

Cette étape finalise le processus de **RAG** en combinant la recherche d’informations pertinentes dans un corpus de texte et la génération de réponses intelligentes basées sur ces informations.

In [13]:
!pip install openai==0.28

  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


In [14]:
import openai


openai.api_key = ""

# Concaténer les top-N segments en un seul contexte
context = "\n".join(top_n_paragraphs)
prompt = f"Contexte :\n{context}\n\nQuestion : {query}\nRéponse :"

# Appeler l'API d'OpenAI
response = openai.ChatCompletion.create(
    model="gpt-4o",  # Utilisation du modèle gpt-4o
    messages=[
        {"role": "system", "content": "Tu es un assistant specialisé dans la recherche d'information à partir de documents fournis. Tes réponses doivent absolument provenir du contexte fourni."},
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

# Extraire et afficher la réponse générée
generated_response = response['choices'][0]['message']['content'].strip()
print(f"Réponse générée : {generated_response}")

Réponse générée : Les LLMs open source mentionnés dans le contexte sont Llama, Mistral, Gemma, DeepSeek, et Qwen.
